# TS CSV (`point_oid`) → ASCII PLY (grouped export)

This notebook reads a **georeferenced Leica/TS CSV** (already in a metric CRS, e.g. `EPSG:3812` ellipsoidal),
filters rows by `point_oid` patterns (wildcards), and exports **separate ASCII PLY files** per group.

### Default example groups
- `CP*`  → e.g. `CP1`, `CP2`, `CP3`
- `1_Auto_*` → all auto-tracked points for route 1

It also optionally exports a TXT sidecar (`point_oid, timestamp, x, y, z`) for each group.


In [ ]:

# ============================================================
# CONFIG
# ============================================================
from pathlib import Path

# Default input (your local Windows path)
INPUT_CSV = r"C:\Users\EB\Desktop\ORBIT2.0\Orbit2\4_RawData\2_ATR\2_TS\200226_MEENDRE2_georef_EPSG3812_ellipsoidal.csv"

# Fallback for this notebook sandbox (used only if INPUT_CSV does not exist locally)
FALLBACK_INPUT_CSV = r"/mnt/data/200226_MEENDRE2_georef_EPSG3812_ellipsoidal.csv"

# Output folder (default: next to input file)
OUTPUT_DIR = None   # e.g. r"C:\Users\EB\Desktop\ORBIT2.0\Orbit2\4_RawData\2_ATR\2_TS\_PLY"

# Column names (adjust only if your CSV changes)
COL_POINT_OID = "point_oid"
COL_TIMESTAMP = "timestamp"
COL_X = "Easting"
COL_Y = "Northing"
COL_Z = "Elevation"

# Wildcard-based export groups (one PLY per group)
# Use shell-style patterns: * and ?
GROUP_SPECS = [
    {
        "name": "CP",
        "patterns": ["CP*"],           # -> CP1, CP2, CP3, ...
    },
    {
        "name": "1_Auto",
        "patterns": ["1_Auto_*"],      # -> 1_Auto_000001, ...
    },
]

# Sorting options
SORT_BY_TIMESTAMP_IF_PRESENT = True   # if timestamp can be parsed
FALLBACK_SORT = "input_order"         # "input_order" or "point_oid"

# Export options
EXPORT_PLY = True
EXPORT_TXT = True
PLY_FLOAT_FORMAT = ".6f"              # coordinate precision
DEDUPLICATE_XYZ = False               # keep False unless you intentionally want to remove duplicate rows

# Optional colors in PLY (MeshLab/Open3D friendly)
# If False -> PLY with x,y,z only
WRITE_RGB = False
DEFAULT_RGB = (255, 255, 255)
GROUP_RGB = {
    "CP": (0, 255, 255),        # cyan
    "1_Auto": (255, 170, 0),    # orange
}

# If True, export one PLY PER EXACT point_oid inside each group (in addition to the group file)
EXPORT_INDIVIDUAL_POINT_OID_FILES = False

# Safety
FAIL_IF_NO_MATCH = False


In [ ]:

# ============================================================
# IMPORTS + UTILITIES
# ============================================================
import os
import re
import fnmatch
from pathlib import Path
import pandas as pd
import numpy as np


def resolve_input_path(path_str: str, fallback_str: str | None = None) -> Path:
    p = Path(path_str)
    if p.exists():
        return p
    if fallback_str:
        fb = Path(fallback_str)
        if fb.exists():
            print(f"[INFO] INPUT_CSV not found locally. Using fallback: {fb}")
            return fb
    raise FileNotFoundError(f"Input CSV not found: {p}")


def read_csv_flexible(path: Path) -> pd.DataFrame:
    """
    Reads CSV robustly (comma/semicolon) and trims column names.
    """
    last_err = None
    for sep in [",", ";", None]:
        try:
            if sep is None:
                df = pd.read_csv(path, sep=None, engine="python")
            else:
                df = pd.read_csv(path, sep=sep)
            df.columns = [str(c).strip() for c in df.columns]
            return df
        except Exception as e:
            last_err = e
    raise RuntimeError(f"Could not read CSV: {path}\nLast error: {last_err}")


def sanitize_name(s: str) -> str:
    s = str(s).strip()
    s = re.sub(r"[^\w\-.]+", "_", s)
    s = re.sub(r"_+", "_", s).strip("_")
    return s or "unnamed"


def ensure_required_columns(df: pd.DataFrame, cols: list[str]):
    missing = [c for c in cols if c not in df.columns]
    if missing:
        raise KeyError(f"Missing required column(s): {missing}\nAvailable: {list(df.columns)}")


def match_any_pattern(values: pd.Series, patterns: list[str]) -> pd.Series:
    """
    Wildcard matching (fnmatch), case-sensitive.
    """
    s = values.astype(str)
    mask = pd.Series(False, index=s.index)
    for pat in patterns:
        mask = mask | s.apply(lambda v, p=pat: fnmatch.fnmatchcase(v, p))
    return mask


def prepare_dataframe(df_raw: pd.DataFrame) -> pd.DataFrame:
    df = df_raw.copy()
    # Preserve original order for stable fallback sorting
    df["_input_order"] = np.arange(len(df))

    # Coerce coords
    for c in [COL_X, COL_Y, COL_Z]:
        df[c] = pd.to_numeric(df[c], errors="coerce")

    # Point oid as string
    df[COL_POINT_OID] = df[COL_POINT_OID].astype(str).str.strip()

    # Parse timestamps (optional)
    if COL_TIMESTAMP in df.columns:
        df["_ts"] = pd.to_datetime(df[COL_TIMESTAMP], errors="coerce")
    else:
        df["_ts"] = pd.NaT

    return df


def sort_group_df(df_group: pd.DataFrame) -> pd.DataFrame:
    if SORT_BY_TIMESTAMP_IF_PRESENT and "_ts" in df_group.columns and df_group["_ts"].notna().any():
        # Timestamp first where available, then preserve input order
        return df_group.sort_values(by=["_ts", "_input_order"], kind="stable")
    if FALLBACK_SORT == "point_oid":
        return df_group.sort_values(by=[COL_POINT_OID, "_input_order"], kind="stable")
    return df_group.sort_values(by=["_input_order"], kind="stable")


def deduplicate_xyz(df_group: pd.DataFrame) -> pd.DataFrame:
    if not DEDUPLICATE_XYZ:
        return df_group
    return df_group.drop_duplicates(subset=[COL_X, COL_Y, COL_Z], keep="first")


def write_ascii_ply_points(path_ply: Path, xyz: np.ndarray, rgb: tuple[int, int, int] | None = None):
    path_ply.parent.mkdir(parents=True, exist_ok=True)
    n = xyz.shape[0]
    use_rgb = rgb is not None and WRITE_RGB

    with open(path_ply, "w", encoding="utf-8", newline="\n") as f:
        f.write("ply\n")
        f.write("format ascii 1.0\n")
        f.write(f"comment generated_by point_oid_filter_notebook\n")
        f.write(f"element vertex {n}\n")
        f.write("property float x\n")
        f.write("property float y\n")
        f.write("property float z\n")
        if use_rgb:
            f.write("property uchar red\n")
            f.write("property uchar green\n")
            f.write("property uchar blue\n")
        f.write("end_header\n")
        if use_rgb:
            r, g, b = [int(v) for v in rgb]
            for x, y, z in xyz:
                f.write(f"{format(float(x), PLY_FLOAT_FORMAT)} {format(float(y), PLY_FLOAT_FORMAT)} {format(float(z), PLY_FLOAT_FORMAT)} {r} {g} {b}\n")
        else:
            for x, y, z in xyz:
                f.write(f"{format(float(x), PLY_FLOAT_FORMAT)} {format(float(y), PLY_FLOAT_FORMAT)} {format(float(z), PLY_FLOAT_FORMAT)}\n")


def write_sidecar_txt(path_txt: Path, df_group: pd.DataFrame):
    path_txt.parent.mkdir(parents=True, exist_ok=True)
    cols = [COL_POINT_OID]
    if COL_TIMESTAMP in df_group.columns:
        cols.append(COL_TIMESTAMP)
    cols += [COL_X, COL_Y, COL_Z]
    out = df_group[cols].copy()
    out.to_csv(path_txt, sep="\t", index=False, encoding="utf-8")


def export_group(df_all: pd.DataFrame, input_csv_path: Path, group_name: str, patterns: list[str], out_dir: Path) -> dict:
    mask = match_any_pattern(df_all[COL_POINT_OID], patterns)
    df_g = df_all.loc[mask].copy()
    df_g = df_g.dropna(subset=[COL_X, COL_Y, COL_Z])
    df_g = sort_group_df(df_g)
    df_g = deduplicate_xyz(df_g)

    result = {
        "group_name": group_name,
        "patterns": patterns,
        "rows": int(len(df_g)),
        "unique_point_oid": int(df_g[COL_POINT_OID].nunique()) if len(df_g) else 0,
        "ply": None,
        "txt": None,
    }

    if len(df_g) == 0:
        return result

    base_stem = sanitize_name(input_csv_path.stem)
    gstem = sanitize_name(group_name)
    ply_path = out_dir / f"{base_stem}__{gstem}.ply"
    txt_path = out_dir / f"{base_stem}__{gstem}.txt"

    xyz = df_g[[COL_X, COL_Y, COL_Z]].to_numpy(dtype=float)
    rgb = GROUP_RGB.get(group_name, DEFAULT_RGB)

    if EXPORT_PLY:
        write_ascii_ply_points(ply_path, xyz, rgb=rgb)
        result["ply"] = str(ply_path)
    if EXPORT_TXT:
        write_sidecar_txt(txt_path, df_g)
        result["txt"] = str(txt_path)

    # Optional per-point_oid split
    if EXPORT_INDIVIDUAL_POINT_OID_FILES:
        sub_dir = out_dir / f"{base_stem}__{gstem}_by_point_oid"
        sub_dir.mkdir(parents=True, exist_ok=True)
        for poid, dfi in df_g.groupby(COL_POINT_OID, sort=False):
            poid_safe = sanitize_name(poid)
            sub_ply = sub_dir / f"{base_stem}__{gstem}__{poid_safe}.ply"
            sub_xyz = dfi[[COL_X, COL_Y, COL_Z]].to_numpy(dtype=float)
            if EXPORT_PLY:
                write_ascii_ply_points(sub_ply, sub_xyz, rgb=rgb)
            if EXPORT_TXT:
                sub_txt = sub_dir / f"{base_stem}__{gstem}__{poid_safe}.txt"
                write_sidecar_txt(sub_txt, dfi)

    return result


def preview_matches(df_all: pd.DataFrame, group_specs: list[dict], max_show: int = 12):
    print("=" * 88)
    print("MATCH PREVIEW")
    print("=" * 88)
    for g in group_specs:
        name = g["name"]
        pats = g["patterns"]
        mask = match_any_pattern(df_all[COL_POINT_OID], pats)
        d = df_all.loc[mask]
        u = d[COL_POINT_OID].dropna().astype(str).unique().tolist()
        print(f"[{name}] patterns={pats}")
        print(f"  rows={len(d):,} | unique point_oid={len(u):,}")
        if u:
            show = u[:max_show]
            suffix = " ..." if len(u) > max_show else ""
            print(f"  sample: {show}{suffix}")
        else:
            print("  sample: <no matches>")
        print("-" * 88)


In [ ]:

# ============================================================
# RUN EXPORT
# ============================================================
input_csv_path = resolve_input_path(INPUT_CSV, FALLBACK_INPUT_CSV)
out_dir = Path(OUTPUT_DIR) if OUTPUT_DIR else input_csv_path.parent
out_dir.mkdir(parents=True, exist_ok=True)

df_raw = read_csv_flexible(input_csv_path)
ensure_required_columns(df_raw, [COL_POINT_OID, COL_X, COL_Y, COL_Z])
if COL_TIMESTAMP not in df_raw.columns:
    print(f"[WARN] '{COL_TIMESTAMP}' column not found. Sorting by {FALLBACK_SORT}.")

df = prepare_dataframe(df_raw)

print(f"[INFO] Input CSV: {input_csv_path}")
print(f"[INFO] Rows: {len(df):,}")
print(f"[INFO] Output dir: {out_dir}")
print(f"[INFO] CRS assumption: file already georeferenced metric (e.g. EPSG:3812 ellipsoidal)")

preview_matches(df, GROUP_SPECS, max_show=10)

results = []
for spec in GROUP_SPECS:
    name = spec["name"]
    pats = spec["patterns"]
    res = export_group(df, input_csv_path, name, pats, out_dir)
    results.append(res)

# Summary table
res_df = pd.DataFrame(results)
print("\n" + "=" * 88)
print("EXPORT SUMMARY")
print("=" * 88)
display(res_df)

# Optional hard fail if nothing matched
if FAIL_IF_NO_MATCH and (res_df["rows"].fillna(0).sum() == 0):
    raise RuntimeError("No rows matched any GROUP_SPECS pattern.")

# Report exact paths
for r in results:
    print("\n" + "-" * 88)
    print(f"Group: {r['group_name']} | rows={r['rows']:,} | unique point_oid={r['unique_point_oid']:,}")
    if r["ply"]:
        print(f"PLY: {r['ply']}")
    if r["txt"]:
        print(f"TXT: {r['txt']}")


## Notes

- The CSV is assumed to already contain **metric coordinates** (`Easting`, `Northing`, `Elevation`) in your target CRS.
- Pattern matching uses **wildcards**:
  - `CP*` matches `CP1`, `CP2`, `CP3`, ...
  - `1_Auto_*` matches `1_Auto_000001`, ...
- To create more exports, just add another entry to `GROUP_SPECS`.
- If you want **one file per exact `point_oid`**, set `EXPORT_INDIVIDUAL_POINT_OID_FILES = True`.


In [ ]:

# ============================================================
# OPTIONAL: inspect all unique point_oid values (top N)
# ============================================================
TOP_N = 200
vc = df[COL_POINT_OID].value_counts(dropna=False)
display(vc.head(TOP_N).rename("count").to_frame())
